# Suite PHY — The physical layout

`Physical` defines the coordinates physical layout gives the logical layout's points and the frames of its diagram, shapes, grids and cells, in diagram units, each an annotation owned through `Logical.Owns` by what it annotates. mbse-basis [CAP-241: physical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-241); its components, which place them, are TypeScript's only.

In [ ]:
from support import Bench, text
from mbse.Schemas.Framework import JSON, Paths, Validators
from mbse.Diagrams.Framework import Container, Logical, Physical

b = Bench()
b.store.transaction(Physical.install)
L, P = b.store.Logical, b.store.Physical


def at(x, y):
    return lambda v: v.x(x).y(y)


def size(width, height):
    return lambda v: v.width(width).height(height)


def annotation(of, name):
    """The owning entry of an annotation of `of`, as `name`."""
    return lambda a: a.insert(lambda e: e.name(name).owner(of))


def frame(of, role, x, y, width, height, unit):
    return P.OfFrame(lambda f: getattr(f.owner(annotation(of, "frame")), role)(lambda a: a.insert(lambda e: getattr(
        e, role)(of))).absolute(at(x, y)).relative(at(x, y)).size(size(width, height)).unit_length(unit))

<a id="PHY-01"></a>
## PHY-01 · The install, after Logical's and Container's, defines the physical layout's schemas, each valid, and run again writes nothing

In [ ]:
NAMES = ['Physical.OfPoint2', 'Physical.OfSize', 'Physical.Placement', 'Physical.OfPlacement', 'Physical.DiagramFrame', 'Physical.ShapeFrame', 'Physical.GridFrame', 'Physical.CellFrame', 'Physical.OfFrame']
ANNOTATED = {"Logical.OfPoint": "placement", "Logical.OfDiagram": "frame", "Logical.OfShape": "frame",
             "Container.OfGrid": "frame", "Container.OfCell": "frame"}
schemas = [b.store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
b.store.transaction(Physical.install)
assert [b.store.registered(name) for name in NAMES] == schemas
assert all(member in b.store.registered(name).adjacencies for name, member in ANNOTATED.items())
assert {"unit_length", "scaling_factor"} <= set(b.store.registered("Logical.OfDiagram").properties)

<a id="PHY-02"></a>
## PHY-02 · A point's placement is owned by the point: its absolute and relative coordinates and its direction, and its path the point's

In [ ]:
assert b.child('', 'cpu') == '[{"kind":"grid","container":"overview","span":3,"grows":"rows","cell_rotation":0.0},{"kind":"place","shape":"cpu","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.layout('cpu', 2) == '[{"kind":"grid","container":"cpu","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
a = b.shape("cpu.a")
with b.store.transaction():
    o = L.OfPoint(lambda p: p.owner(annotation(a, "origin")).origin_of(lambda x: x.insert(lambda e: e.shape(a))))
    P.OfPlacement(lambda q: q.owner(annotation(o, "placement")).point(lambda x: x.insert(lambda e: e.point(o)))
                  .absolute(at(4.0, 3.0)).relative(at(4.0, 3.0)).direction(0.0))
placement = o.owns({"name": "placement"})
path = lambda x: JSON.ToJSON(b.store)(b.store.Paths.OfRoot.as_schema(), Paths.of(x))
assert text([placement.absolute.x.get(), placement.absolute.y.get(), placement.relative.x.get(), placement.direction.get()]) == '[4.0,3.0,4.0,0.0]'
assert text([[e.placement is placement for e in o.placement], [e.point is o for e in placement.point]]) == '[[true],[true]]'
assert text(path(placement)[len(path(o)) - 13:]) == '": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"placement\\"}, \\"next\\": {\\"target\\": {}}}}}}}}}}}"'
assert text(JSON.ToJSON(b.store)(P.OfPlacement.as_schema(), placement)) == '"{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"0\\": {\\"schema\\": [{\\"$ref\\": {\\"singleton\\": \\".\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"Physical\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"OfPlacement\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, \\"spec\\": [{\\"0\\": [{\\"as_property\\": {\\"name\\": \\"absolute\\", \\"mutations\\": [{\\"set\\": {\\"value\\": {\\"x\\": 4.0, \\"y\\": 3.0}}}]}}, {\\"as_property\\": {\\"name\\": \\"relative\\", \\"mutations\\": [{\\"set\\": {\\"value\\": {\\"x\\": 4.0, \\"y\\": 3.0}}}]}}, {\\"as_property\\": {\\"name\\": \\"direction\\", \\"mutations\\": [{\\"set\\": {\\"value\\": 0.0}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"owner\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"owner\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"Logical.Diagrams\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"overview\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"cpu.a\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"origin\\"}, \\"next\\": {\\"target\\": {}}}}}}}}}}}}]}}, {\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"placement\\"}}]}}]}]}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"point\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"point\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"Logical.Diagrams\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"overview\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"cpu.a\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"origin\\"}, \\"next\\": {\\"target\\": {}}}}}}}}}}}}]}}]}]}}]}}]}]}]}}]}"'

<a id="PHY-03"></a>
## PHY-03 · The frames of a diagram, a shape, a grid and a cell, each with its unit length; the diagram's tuning

In [ ]:
cpu = b.shape("cpu")
((entry,),) = [list(cpu.grid)]
grid = entry.grid
(cell,) = [e.cell for e in grid.cells]
with b.store.transaction():
    L.OfDiagram(b.overview, lambda d: d.unit_length(1.0).scaling_factor(0.5))
    frame(b.overview, "diagram", 0.0, 0.0, 6.0, 4.0, 1.0)
    frame(cpu, "shape", 0.0, 0.0, 2.0, 1.0, 0.5)
    frame(grid, "grid", 0.0, 0.0, 2.0, 1.0, 0.5)
    frame(cell, "cell", 0.0, 0.0, 1.0, 1.0, 0.5)
shown = [x.owns({"name": "frame"}) for x in (b.overview, cpu, grid, cell)]
assert text([[f.size.width.get(), f.size.height.get(), f.unit_length.get()] for f in shown]) == '[[6.0,4.0,1.0],[2.0,1.0,0.5],[2.0,1.0,0.5],[1.0,1.0,0.5]]'
assert text([b.overview.unit_length.get(), b.overview.scaling_factor.get(), [e.frame is shown[0] for e in b.overview.frame]]) == '[1.0,0.5,[true]]'
assert text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) == '[]'

<a id="PHY-04"></a>
## PHY-04 · One placement per point, as the validator reports

In [ ]:
with b.store.transaction():
    P.OfPlacement(lambda q: q.owner(annotation(o, "another")).point(lambda x: x.insert(lambda e: e.point(o))).direction(90.0))
assert text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) == '["unique(placement) violated: entries agreeing on [\'point\'] differ on [\'placement\']"]'

<a id="PHY-05"></a>
## PHY-05 · An annotation goes with what it annotates: a child removed from its container keeps its points' placements, for it is not deleted, and the cell it left takes its frame

In [ ]:
def reached(schema):
    """How many objects of `schema` the store reaches from its singletons."""
    return len(b.store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
        lambda p: p.symbols({"x": schema})]))])))
assert text([reached(P.OfPlacement), reached(P.OfFrame), [e.name.get() for e in cpu.owns]]) == '[2,4,["grid","frame"]]'
assert b.unchild('cpu', 'a') == '[{"kind":"leave","shape":"a","from":[0,0]},{"kind":"extents","rows":0,"columns":0},{"kind":"place","shape":"a","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert text([reached(P.OfPlacement), reached(P.OfFrame), [e.name.get() for e in cpu.owns], len(list(grid.cells))]) == '[2,3,["grid","frame"],0]'